# Run the locked larger-development experiment

This notebook runs the development-only GPU experiment after Notebook 05 has validated the lock. It performs the five-condition inference, correctness/calibration analysis, and the locked 64-item faithfulness stage. Raw outputs are written to Google Drive so rerunning the notebook after a Colab disconnect resumes completed work.

**Before starting:** commit and push all larger-development implementation files, select a **T4 GPU** runtime, place `controlled_training_bundle-4.zip` in Google Drive (or upload it when prompted), and set the `HF_TOKEN` Colab secret. The official test partition remains sealed throughout.

In [ ]:
# 1. Confirm the GPU and mount persistent Google Drive storage.
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime: Runtime > Change runtime type > T4 GPU")
gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)
if "T4" not in gpu_name:
    raise RuntimeError(f"The locked protocol requires a T4 GPU, observed: {gpu_name}")

drive.mount("/content/drive")
from pathlib import Path
PERSISTENT_ROOT = Path("/content/drive/MyDrive/gi_vqa_study1")
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent experiment directory:", PERSISTENT_ROOT)

In [ ]:
# 2. Clone a fresh checkout and select the exact pushed implementation commit.
# Replace this placeholder with the full 40-character SHA from `git rev-parse HEAD`.
REPOSITORY_COMMIT = "REPLACE_WITH_FULL_PUSHED_COMMIT_SHA"
REPOSITORY_URL = "SET_STANDALONE_REPOSITORY_URL"
REPOSITORY_ROOT = Path("/content/gi-vqa-larger-development")
PROJECT_ROOT = REPOSITORY_ROOT
RUN_DIR = PERSISTENT_ROOT / f"larger-development-{REPOSITORY_COMMIT}"

import re
import shutil
import subprocess

if not re.fullmatch(r"[0-9a-f]{40}", REPOSITORY_COMMIT):
    raise ValueError("Set REPOSITORY_COMMIT to the full pushed 40-character commit SHA")
if REPOSITORY_ROOT.exists():
    shutil.rmtree(REPOSITORY_ROOT)
subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True)
subprocess.run(["git", "checkout", "--detach", REPOSITORY_COMMIT], cwd=REPOSITORY_ROOT, check=True)
observed = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_ROOT, check=True,
    capture_output=True, text=True,
).stdout.strip()
status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPOSITORY_ROOT, check=True,
    capture_output=True, text=True,
).stdout
if observed != REPOSITORY_COMMIT or status:
    raise RuntimeError("Exact clean checkout verification failed")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Commit:", observed)
print("Persistent run directory:", RUN_DIR)
%cd /content/gi-vqa-larger-development

In [ ]:
# 3. Install the pinned GPU and analysis dependencies.
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "-e", ".[gpu]", "fsspec==2024.12.0",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)
print("Pinned dependency installation completed")

In [ ]:
# 4. Verify the runtime versions enforced by the protocol.
import importlib.metadata as metadata
import sys
import torch

expected = {
    "accelerate": "1.9.0", "bitsandbytes": "0.47.0", "datasets": "3.3.2",
    "huggingface-hub": "0.34.3", "ms-swift": "3.7.0", "numpy": "2.0.2",
    "peft": "0.16.0", "Pillow": "11.3.0", "PyYAML": "6.0.2",
    "sentencepiece": "0.2.0", "transformers": "4.55.0", "wandb": "0.21.0",
}
mismatches = {}
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
for package, required in expected.items():
    observed = metadata.version(package)
    print(f"{package}: {observed}")
    if observed != required:
        mismatches[package] = {"expected": required, "observed": observed}
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Python 3.11 is required")
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("A CUDA T4 is required")
if not str(torch.__version__).startswith("2.6.0"):
    raise RuntimeError(f"Expected PyTorch 2.6.0, observed {torch.__version__}")
if mismatches:
    raise RuntimeError(f"Package mismatches: {mismatches}")
print("Pinned runtime PASS")

In [ ]:
# 5. Authenticate to Hugging Face without printing or storing the token.
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download, login, whoami

hf_token = userdata.get("HF_TOKEN")
if not isinstance(hf_token, str) or not hf_token.strip():
    raise RuntimeError("Add HF_TOKEN in Colab Secrets and enable notebook access")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
identity = whoami(token=hf_token)
print("Authenticated as:", identity.get("name") or identity.get("fullname"))
access_path = hf_hub_download(
    repo_id="google/paligemma-3b-pt-224",
    filename="config.json",
    revision="35e4f46485b4d07967e7e9935bc3786aad50687c",
    token=hf_token,
)
print("PaliGemma access PASS:", access_path)
del hf_token

In [ ]:
# 6. Locate the training bundle in Drive, or upload it once and persist it there.
from google.colab import files

BUNDLE_ARCHIVE = PERSISTENT_ROOT / "controlled_training_bundle-4.zip"
if not BUNDLE_ARCHIVE.is_file():
    print("Select controlled_training_bundle-4.zip")
    uploaded = files.upload()
    payload = uploaded.get("controlled_training_bundle-4.zip")
    if payload is None:
        raise RuntimeError("Upload must be named controlled_training_bundle-4.zip")
    BUNDLE_ARCHIVE.write_bytes(payload)
print("Persistent bundle archive:", BUNDLE_ARCHIVE)
print("Archive bytes:", BUNDLE_ARCHIVE.stat().st_size)

In [ ]:
# 7. Safely extract and validate the ignored training bundle inside the checkout.
import zipfile

TRAINING_BUNDLE = PROJECT_ROOT / "controlled_training_bundle-4"
TRAINING_BUNDLE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_ARCHIVE) as archive:
    for member in archive.infolist():
        target = (TRAINING_BUNDLE / member.filename).resolve()
        target.relative_to(TRAINING_BUNDLE.resolve())
    archive.extractall(TRAINING_BUNDLE)
required = [
    TRAINING_BUNDLE / "bundle_manifest.json",
    TRAINING_BUNDLE / "controlled_training_report.json",
    TRAINING_BUNDLE / "prepared_data/constant_image.png",
    TRAINING_BUNDLE / "adapters/paired_image/adapter_model.safetensors",
    TRAINING_BUNDLE / "adapters/constant_image/adapter_model.safetensors",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError(f"Training bundle is incomplete: {missing}")
print("Training bundle PASS:", TRAINING_BUNDLE)

In [ ]:
# 8. Reconstruct the ignored development split and validate the locked protocol.
import os

environment = os.environ.copy()
environment["PYTHONPATH"] = "src"
materialize = subprocess.run(
    [
        sys.executable, "-m", "gi_vqa.cli", "materialize-splits",
        "--manifest", "protocols/study1/grouped_split_manifest.json",
        "--project-root", ".",
    ],
    cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True,
)
print(materialize.stdout)
if materialize.returncode:
    print(materialize.stderr)
    raise RuntimeError("Development split materialization failed")
subprocess.run(
    [
        sys.executable, "-m", "gi_vqa.larger_development", "check",
        "--project-root", ".",
        "--protocol", "protocols/study1/larger_development_protocol.json",
    ],
    cwd=PROJECT_ROOT, env=environment, check=True,
)
status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPOSITORY_ROOT, check=True,
    capture_output=True, text=True,
).stdout
if status:
    raise RuntimeError(f"Generated inputs unexpectedly dirtied the checkout:\n{status}")
print("Protocol PASS and checkout remains clean")

## Long-running stages

Cells 9 and 12 are restart-safe. If Colab disconnects, reconnect, rerun Cells 1–8, and rerun the interrupted cell. Completed item files in Google Drive are validated and reused. Do not delete or edit the persistent run directory between resumptions.

In [ ]:
# 9. Run or resume all 256 items under the five locked inference conditions.
inference_command = [
    sys.executable, "-m", "gi_vqa.larger_development_runner",
    "--project-root", ".",
    "--protocol", "protocols/study1/larger_development_protocol.json",
    "--training-bundle", str(TRAINING_BUNDLE.relative_to(PROJECT_ROOT)),
    "--run-dir", str(RUN_DIR),
    "--expected-commit", REPOSITORY_COMMIT,
    "--require-clean-git",
    "--required-gpu-substring", "T4",
]
print("Starting/resuming larger inference. Outputs persist in:", RUN_DIR)
completed = subprocess.run(inference_command, cwd=PROJECT_ROOT, env=environment)
if completed.returncode != 0:
    raise RuntimeError(
        f"Inference exited with code {completed.returncode}. Inspect the traceback; "
        "after fixing an environmental interruption, rerun this cell to resume."
    )
print("Larger-development inference process completed")

In [ ]:
# 10. Require complete five-condition inference before analysis.
import json

inference_status_path = RUN_DIR / "inference_status.json"
inference_status = json.loads(inference_status_path.read_text(encoding="utf-8"))
print(json.dumps({
    "status": inference_status.get("status"),
    "completed_item_conditions": inference_status.get("completed_item_conditions"),
    "expected_item_conditions": inference_status.get("expected_item_conditions"),
    "test_partition_accessed": inference_status.get("test_partition_accessed"),
}, indent=2))
if inference_status.get("status") != "INFERENCE_COMPLETE":
    raise RuntimeError("Inference is incomplete; rerun Cell 9")
if inference_status.get("test_partition_accessed") is not False:
    raise RuntimeError("Test-set seal verification failed")

In [ ]:
# 11. Compute locked correctness, grounding and calibration analysis.
subprocess.run(
    [
        sys.executable, "-m", "gi_vqa.larger_development_analysis",
        "--project-root", ".",
        "--protocol", "protocols/study1/larger_development_protocol.json",
        "--run-dir", str(RUN_DIR),
    ],
    cwd=PROJECT_ROOT, env=environment, check=True,
)
analysis_path = RUN_DIR / "larger_development_analysis.json"
analysis = json.loads(analysis_path.read_text(encoding="utf-8"))
print(json.dumps({
    "status": analysis.get("status"),
    "promotion_decision": analysis.get("promotion_decision"),
    "test_partition_accessed": analysis.get("test_partition_accessed"),
}, indent=2))

In [ ]:
# 12. Run or resume the locked 64-item paired-vs-constant faithfulness stage.
faithfulness_command = [
    sys.executable, "-m", "gi_vqa.larger_development_faithfulness",
    "--project-root", ".",
    "--protocol", "protocols/study1/larger_development_protocol.json",
    "--training-bundle", str(TRAINING_BUNDLE.relative_to(PROJECT_ROOT)),
    "--inference-run-dir", str(RUN_DIR),
    "--expected-commit", REPOSITORY_COMMIT,
    "--require-clean-git",
    "--required-gpu-substring", "T4",
]
print("Starting/resuming faithfulness evaluation")
completed = subprocess.run(faithfulness_command, cwd=PROJECT_ROOT, env=environment)
if completed.returncode != 0:
    raise RuntimeError(
        f"Faithfulness exited with code {completed.returncode}. Inspect the traceback; "
        "after fixing an environmental interruption, rerun this cell to resume."
    )
print("Faithfulness process completed")

In [ ]:
# 13. Validate final statuses and create a hashed evidence manifest.
import hashlib

faithfulness_path = RUN_DIR / "faithfulness/faithfulness_report.json"
faithfulness = json.loads(faithfulness_path.read_text(encoding="utf-8"))
if faithfulness.get("status") != "PASS":
    raise RuntimeError("Faithfulness stage is incomplete; rerun Cell 12")
if faithfulness.get("test_partition_accessed") is not False:
    raise RuntimeError("Faithfulness test-set seal verification failed")

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

evidence = {
    "repository_commit": REPOSITORY_COMMIT,
    "test_partition_accessed": False,
    "artifacts": {
        "inference_status.json": sha256(inference_status_path),
        "larger_development_analysis.json": sha256(analysis_path),
        "faithfulness/faithfulness_report.json": sha256(faithfulness_path),
    },
}
evidence_path = RUN_DIR / "final_evidence_manifest.json"
evidence_path.write_text(json.dumps(evidence, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(evidence, indent=2))

In [ ]:
# 14. Package and download the complete evidence directory.
import shutil
from google.colab import files

archive_base = PERSISTENT_ROOT / f"larger-development-evidence-{REPOSITORY_COMMIT}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR))
print("Created:", archive_path)
print("SHA-256:", sha256(archive_path))
files.download(str(archive_path))